In [1]:
from dataclasses import dataclass
from enum import Enum
from typing import Any, Callable, Dict, Optional, Type
from pydantic import BaseModel, ConfigDict, Field, ValidationError


# ==========================================
# 1. 统一 Error Taxonomy (扩展 Contract 错误)
# ==========================================
class ErrorKind(Enum):
    INPUT_VALIDATION_ERROR = "input_validation_error"    # LLM/Caller 参数合规性失败 (责任在 LLM)
    OUTPUT_VALIDATION_ERROR = "output_validation_error"  # Backend/Adapter 数据契约损坏 (责任在 Backend/Tool)
    RUNTIME_EXECUTION_ERROR = "runtime_execution_error"  # 运行时物理执行失败 (网络/超时等)


@dataclass
class ToolError:
    kind: ErrorKind
    message: str
    retryable: bool


@dataclass
class ToolResult:
    ok: bool
    tool_name: str
    data: Any = None
    error: Optional[ToolError] = None


# ==========================================
# 2. Schema 定义 (Input & Output Contracts)
# ==========================================
STRICT_CONFIG = ConfigDict(extra="forbid")

# 1. Input Contract: 保护 Tool Runtime 不被非法参数轰炸
class GetLeaveBalanceArgs(BaseModel):
    """查询指定员工的剩余带薪假期天数。"""
    model_config = STRICT_CONFIG

    employee_id: str = Field(
        ...,
        pattern=r"^EMP_\d{4}$",
        description="员工唯一编号，格式为 'EMP_' 加上 4 位数字，例如 'EMP_1001'",
    )


# 2. Output Contract: 保护 Agent 接收干净、受控、最小暴露面且强类型的数据
class LeaveBalanceOutput(BaseModel):
    """向 Agent 暴露的标准带薪假数据契约。"""
    model_config = STRICT_CONFIG

    employee_id: str = Field(..., description="标准员工编号")
    remaining_days: float = Field(..., ge=0.0, description="剩余带薪假天数（天）")


# ==========================================
# 3. Tool Config & Registry
# ==========================================
class ToolConfig:
    def __init__(
        self,
        name: str,
        input_schema_cls: Type[BaseModel],
        output_schema_cls: Type[BaseModel],
    ):
        self.name = name
        self.input_schema_cls = input_schema_cls
        self.output_schema_cls = output_schema_cls


class ToolRegistry:
    def __init__(self):
        self.configs: Dict[str, ToolConfig] = {}
        self.funcs: Dict[str, Callable] = {}

    def register(self, config: ToolConfig, func: Callable):
        self.configs[config.name] = config
        self.funcs[config.name] = func

    def get_config(self, name: str) -> ToolConfig:
        return self.configs[name]

    def get_func(self, name: str) -> Callable:
        return self.funcs[name]


# ==========================================
# 4. ToolRuntime & Dispatcher (含双向 Guard)
# ==========================================
class ToolRuntime:
    """负责工具物理执行、Adapter 防腐隔离与 Output Contract 强校验"""
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def execute(self, tool_name: str, validated_input: BaseModel) -> ToolResult:
        config = self.registry.get_config(tool_name)
        tool_fn = self.registry.get_func(tool_name)

        # Step A: 物理执行真实 Tool / Backend API (获取 Raw Backend Result)
        try:
            raw_backend_result = tool_fn(validated_input)
        except Exception as e:
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                error=ToolError(
                    kind=ErrorKind.RUNTIME_EXECUTION_ERROR,
                    message=f"Backend execution exception: {str(e)}",
                    retryable=True,
                ),
            )

        # Step B: Adapter 机制 & Output Contract 校验
        # 1) Adapter 将后端杂乱或变动的字段映射转换
        # 2) 校验是否符合强类型的 Output Contract (如只暴露 minimum data, 过滤敏感字段)
        try:
            if isinstance(raw_backend_result, config.output_schema_cls):
                validated_output = raw_backend_result
            else:
                validated_output = config.output_schema_cls.model_validate(raw_backend_result)

            return ToolResult(ok=True, tool_name=tool_name, data=validated_output)

        except ValidationError as e:
            # 数据契约损坏 -> OUTPUT_VALIDATION_ERROR (阻止脏数据进入 Agent 上下文)
            error_details = "; ".join([f"{err['loc'][0]}: {err['msg']}" for err in e.errors()])
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                error=ToolError(
                    kind=ErrorKind.OUTPUT_VALIDATION_ERROR,
                    message=f"Output Contract Validation Failed -> {error_details}",
                    retryable=False,  # 后端脏数据，重试同一输入无意义
                ),
            )


class AgentToolDispatcher:
    """入口分发器：控制 Input Contract 校验"""
    def __init__(self, registry: ToolRegistry, runtime: ToolRuntime):
        self.registry = registry
        self.runtime = runtime

    def dispatch(self, tool_name: str, raw_args: Dict[str, Any]) -> ToolResult:
        try:
            config = self.registry.get_config(tool_name)
        except KeyError:
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                error=ToolError(
                    kind=ErrorKind.INPUT_VALIDATION_ERROR,
                    message=f"Unknown tool name: '{tool_name}'",
                    retryable=False,
                ),
            )

        # Step 1: Input Validation
        try:
            validated_input = config.input_schema_cls.model_validate(raw_args)
        except ValidationError as e:
            # 入参不符合规范 -> INPUT_VALIDATION_ERROR (拦截非法请求进入 Runtime)
            error_details = "; ".join([f"{err['loc'][0]}: {err['msg']}" for err in e.errors()])
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                error=ToolError(
                    kind=ErrorKind.INPUT_VALIDATION_ERROR,
                    message=f"Input Contract Validation Failed -> {error_details}",
                    retryable=True,  # 允许 Agent 根据 Validation Error 修正参数后重试
                ),
            )

        # Step 2: 进入 ToolRuntime
        return self.runtime.execute(tool_name, validated_input)


# ==========================================
# 5. Tool / Backend 实现 & Adapter 防腐层
# ==========================================
# 全局标志位：用于 Mock Scenario 3 的脏数据场景
BACKEND_RETURN_CORRUPTED_DATA = False

def backend_get_leave_balance_tool(args: GetLeaveBalanceArgs) -> Dict[str, Any]:
    """
    模拟真实的 HR Backend API。
    特点：
    1. 内部数据库字段为 staff_no 和 leave_days
    2. 返回许多冗余/敏感信息 (salary, internal_rating)
    3. 在场景 3 中会返回污染数据 "unknown"
    """
    if BACKEND_RETURN_CORRUPTED_DATA:
        raw_hr_api_response = {
            "staff_no": args.employee_id,
            "leave_days": "unknown",  # 脏数据！损坏的类型
            "salary_band": "L5",
            "internal_rating": "A+",
        }
    else:
        raw_hr_api_response = {
            "staff_no": args.employee_id,
            "leave_days": 7.5,
            "salary_band": "L5",
            "internal_rating": "A+",
        }

    # Adapter 层：阻断 Backend 字段变化与敏感暴露，只提取 & 转换成 Agent 契约要求的字段
    adapted_result = {
        "employee_id": raw_hr_api_response["staff_no"],
        "remaining_days": raw_hr_api_response["leave_days"],
        # 敏感/无用字段（salary_band, internal_rating）被 Adapter 直接剪裁，符合最小暴露原则
    }
    return adapted_result


def setup_system() -> AgentToolDispatcher:
    registry = ToolRegistry()
    registry.register(
        ToolConfig(
            name="get_leave_balance",
            input_schema_cls=GetLeaveBalanceArgs,
            output_schema_cls=LeaveBalanceOutput,
        ),
        backend_get_leave_balance_tool,
    )
    runtime = ToolRuntime(registry)
    return AgentToolDispatcher(registry, runtime)


# ==========================================
# 6. 场景验证 (Three Critical Scenarios)
# ==========================================
if __name__ == "__main__":
    dispatcher = setup_system()

    print("=========================================================")
    print("Scenario 1: 合法请求 + 正常 Backend -> ToolResult.ok = True")
    print("=========================================================")
    BACKEND_RETURN_CORRUPTED_DATA = False
    res1 = dispatcher.dispatch("get_leave_balance", {"employee_id": "EMP_1001"})
    print(f"Result OK: {res1.ok}")
    print(f"Returned Data Type: {type(res1.data).__name__}")
    print(f"Data: {res1.data}")
    assert res1.ok
    assert isinstance(res1.data, LeaveBalanceOutput)
    assert res1.data.remaining_days == 7.5
    # 验证 Agent 只拿到了精简的 Contract，绝对看不到 salary_band 等冗余/敏感数据
    assert not hasattr(res1.data, "salary_band")

    print("\n=========================================================")
    print("Scenario 2: 非法入参 (employee_id='Kevin') -> INPUT_VALIDATION_ERROR")
    print("=========================================================")
    res2 = dispatcher.dispatch("get_leave_balance", {"employee_id": "Kevin"})
    print(f"Result OK: {res2.ok}")
    print(f"Error Kind: {res2.error.kind.value}")
    print(f"Error Message: {res2.error.message}")
    assert not res2.ok
    assert res2.error.kind == ErrorKind.INPUT_VALIDATION_ERROR

    print("\n=========================================================")
    print("Scenario 3: 脏数据返回 (leave_days='unknown') -> OUTPUT_VALIDATION_ERROR")
    print("=========================================================")
    BACKEND_RETURN_CORRUPTED_DATA = True
    res3 = dispatcher.dispatch("get_leave_balance", {"employee_id": "EMP_1001"})
    print(f"Result OK: {res3.ok}")
    print(f"Error Kind: {res3.error.kind.value}")
    print(f"Error Message: {res3.error.message}")
    assert not res3.ok
    assert res3.error.kind == ErrorKind.OUTPUT_VALIDATION_ERROR

    print("\n✅ All Day 33 Dual Contract & Adapter Isolation Tests Passed!")

Scenario 1: 合法请求 + 正常 Backend -> ToolResult.ok = True
Result OK: True
Returned Data Type: LeaveBalanceOutput
Data: employee_id='EMP_1001' remaining_days=7.5

Scenario 2: 非法入参 (employee_id='Kevin') -> INPUT_VALIDATION_ERROR
Result OK: False
Error Kind: input_validation_error
Error Message: Input Contract Validation Failed -> employee_id: String should match pattern '^EMP_\d{4}$'

Scenario 3: 脏数据返回 (leave_days='unknown') -> OUTPUT_VALIDATION_ERROR
Result OK: False
Error Kind: output_validation_error
Error Message: Output Contract Validation Failed -> remaining_days: Input should be a valid number, unable to parse string as a number

✅ All Day 33 Dual Contract & Adapter Isolation Tests Passed!
